# 参考题解：HA-LoRA 多头注意力

实现带 Head-Aware LoRA 与头门控的多头自注意力。

核心思路：为每个注意力头分别设置低秩 A/B，并用 softmax 门控调节每个头的 LoRA 更新；基础 Q/K/V/O 投影保持标准 MHA。


In [ ]:
# ✅ SOLUTION

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class PurifiedMultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, rank: int = 8, alpha: float = 1.0):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.d_model, self.num_heads, self.rank = d_model, num_heads, rank
        self.head_dim = d_model // num_heads
        self.scale = alpha / rank
        self.W_q = nn.Linear(d_model,d_model); self.W_k = nn.Linear(d_model,d_model)
        self.W_v = nn.Linear(d_model,d_model); self.W_o = nn.Linear(d_model,d_model)
        self.lora_A_q = nn.Parameter(torch.randn(num_heads,self.head_dim,rank) * 0.02)
        self.lora_B_q = nn.Parameter(torch.zeros(num_heads,rank,self.head_dim))
        self.lora_A_v = nn.Parameter(torch.randn(num_heads,self.head_dim,rank) * 0.02)
        self.lora_B_v = nn.Parameter(torch.zeros(num_heads,rank,self.head_dim))
        self.head_gates = nn.Parameter(torch.ones(num_heads))

    def forward(self, x: torch.Tensor, mask=None):
        b,s,_=x.shape; h,d=self.num_heads,self.head_dim
        xh=x.view(b,s,h,d).transpose(1,2)
        q=self.W_q(x).view(b,s,h,d).transpose(1,2)
        k=self.W_k(x).view(b,s,h,d).transpose(1,2)
        v=self.W_v(x).view(b,s,h,d).transpose(1,2)
        gates=F.softmax(self.head_gates,dim=0).view(1,h,1,1)*h
        q=q+(xh@self.lora_A_q@self.lora_B_q)*self.scale*gates
        v=v+(xh@self.lora_A_v@self.lora_B_v)*self.scale*gates
        scores=q@k.transpose(-2,-1)/math.sqrt(d)
        if mask is not None: scores=scores.masked_fill(mask==0,float("-inf"))
        out=torch.softmax(scores,-1)@v
        return self.W_o(out.transpose(1,2).contiguous().view(b,s,self.d_model))
